# Decay chains and inventories

Bateman chains, branching, and multi-nuclide inventories: watch daughter
nuclides build up, then decay away, with everything driven by the bundled
ICRP-107 half-lives.

**Goals**

1. Solve the classic two-species chain Sr-90 -> Y-90.
2. Evaluate atom counts and activities at a chosen time.
3. Explore a branching (star) chain.
4. Track a mixed inventory with `Inventory`.

**Sources**

* H. Bateman, "The solution of a system of differential equations
  occurring in the theory of radioactive transformation",
  Proc. Cambridge Philos. Soc. 15 (1910) 423-427.
* ICRP Publication 107 (half-lives): https://www.icrp.org/publication.asp?id=ICRP+Publication+107


## Linear chains (Bateman)

For a parent $P \to D_1 \to D_2 \dots$ with decay constants $\lambda_i$,
the Bateman equations give each population in closed form. `DecayChain`
solves them numerically from the bundled half-lives; you supply isotope
names (or raw lambdas) and the starting atoms.

`chain.at(t, n0=...)` returns **atoms** of every species at time `t`;
`chain.activity(t, n0=...)` returns **becquerel** (lambda times atoms).


In [ ]:
import math

from pydecay import DecayChain, Inventory, Nuclide

chain = DecayChain.from_isotopes(["Sr-90", "Y-90"])
t_30y = 30 * 31557600  # 30 Julian years

atoms = chain.at(t_30y, n0={"Sr-90": 1.0})
acts = chain.activity(t_30y, n0={"Sr-90": 1.0})
print("after 30 years, starting from 1 Sr-90 atom:")
for name in ("Sr-90", "Y-90"):
    print(f"  {name}: {atoms[name]:.12e} atoms, {acts[name]:.6e} Bq")


## Branching and inventories

`DecayChain.branching(parent, fractions, lambdas=...)` builds a star: one
parent feeding terminal daughters by fraction (fractions must sum to
at most 1). `Inventory` holds real mixtures in Bq (or atoms/grams):
`inv.decay(t)` propagates every seed **with progeny ingrowth**, and
accessors report atoms, activities, masses, or even dose rates.

A note on Y-90: because the chain was seeded with atoms, the Y-90 activity
after 30 years comes from its own decay constant times its grown-in
population - the Bateman solution at work.


In [ ]:
star = DecayChain.branching(
    "P",
    {"D1": 0.6, "D2": 0.4},
    lambdas={"P": 0.1, "D1": 0.05, "D2": 0.02},  # synthetic, 1/s
)
star_atoms = star.at(10.0, n0={"P": 1.0})
print("branching star at t = 10 s:")
for name, n in star_atoms.items():
    print(f"  {name}: {n:.12e}")

inv = Inventory({"I-131": 1e6, "Tc-99m": 2.5e5}, units="Bq")
later = inv.decay("24 hours")
print()
print("mixed inventory after 24 hours (Bq):")
for name, a in later.activities().items():
    print(f"  {name}: {a:.6e} Bq")
print(f"  total: {later.total_activity():.6e} Bq")


## Exercise: Y-90 activity after 30 years

Starting from 1 atom of Sr-90, how many **becquerel** of Y-90 are there
after 30 Julian years? Compute it from the chain's atom count times the
Y-90 decay constant (`Nuclide.load("Y-90").lambda_`), and cross-check with
`chain.activity`.

The solution is in the next cell, tagged `answer`.


In [ ]:
# ANSWER (cell tagged "answer" - hide while attempting the exercise)
n_y90 = chain.at(t_30y, n0={"Sr-90": 1.0})["Y-90"]
a_y90 = Nuclide.load("Y-90").lambda_ * n_y90
a_y90_via_chain = chain.activity(t_30y, n0={"Sr-90": 1.0})["Y-90"]
print(f"lambda x N = {a_y90:.9e} Bq")
print(f"chain.activity = {a_y90_via_chain:.9e} Bq")
assert math.isclose(a_y90, a_y90_via_chain, rel_tol=1e-12)
print("exercise checks passed")


## Golden check

Pinned in `tests/golden/golden_values.json`: the Sr-90/Y-90 atom counts
after 30 Julian years (Bateman solution with ICRP-107 half-lives) and the
synthetic branching-star tuple at t = 10 s. `tests/test_notebooks.py`
compares every value printed below against that file.


In [ ]:
import json
import math

sr90 = chain.at(t_30y, n0={"Sr-90": 1.0})
star10 = star.at(10.0, n0={"P": 1.0})

assert math.isclose(sr90["Sr-90"], 0.4856441599456503, rel_tol=1e-9)
assert math.isclose(sr90["Y-90"], 0.0001233796088024281, rel_tol=1e-9)
assert math.isclose(star10["P"], 0.36787944117144233, rel_tol=1e-9)
assert math.isclose(star10["D1"], 0.2863814622494293, rel_tol=1e-9)
assert math.isclose(star10["D2"], 0.2254256559532698, rel_tol=1e-9)

payload = {
    "sr90_y90_atoms_30y": {"Sr-90": sr90["Sr-90"], "Y-90": sr90["Y-90"]},
    "branch_p_d1_d2_10s": {k: star10[k] for k in ("P", "D1", "D2")},
}
print("GOLDEN_CHECK " + json.dumps(payload))
